# AIME — Video Generation with ByteDance Seedance (2.5)

Generates videos from the 57 video-compatible AIME prompts (`Modality` in `{'Video','Both'}`) using **ByteDance Seedance**
via the **BytePlus ModelArk** video-generation API (Volcano Engine Ark international route).

**API workflow (async job pattern), per the ModelArk "Video generation API" reference**
(https://docs.byteplus.com/en/docs/ModelArk/Video_Generation_API):
1. `POST /api/v3/contents/generations/tasks` — submit the job (`model`, `content`, `resolution`, `duration`, `ratio`), get back a task id.
2. `GET /api/v3/contents/generations/tasks/{id}` — poll until `status` is `succeeded`/`failed`.
3. Download the resulting MP4 from the response (`content.video_url`); ModelArk cleans up hosted URLs after ~24h, so copy
   the file locally right away (this notebook does that automatically).

**Output structure:**
```
dataset/media/videos/Seedance/
    P001_amb_<hash>.mp4
    ...
```
A CSV index is written to `dataset/labels/generated_Seedance.csv`.

**Prerequisites:**
```bash
pip install requests pandas openpyxl tqdm python-dotenv
```

**Credentials required in `.env`:** `SEEDANCE_API_KEY` (see `.env.example`). Generate one in the BytePlus/Volcengine
console under ModelArk → API Keys. If it is not set, the generation-loop cell will print a warning and skip making any
API calls.

**A note on the exact model id.** ByteDance announced Seedance 2.5 GA for "early July 2026" (this notebook was written
2026-07-02). At research time the only *publicly confirmed* ModelArk model ids were for Seedance 2.0
(`dreamina-seedance-2-0-260128`, `...-fast-260128`, `...-mini-260615`) — the 2.5 id was not yet published in crawlable
docs. `SEEDANCE_MODEL_ID` below is therefore **overridable via `.env`** (`SEEDANCE_MODEL_ID`) so you can drop in the
real 2.5 id from your ModelArk console the moment it's available, without touching code. It currently defaults to the
2.0 id so the notebook is runnable today; **check the console and update `.env` for a true 2.5 run.**

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv
load_dotenv()

SEEDANCE_API_KEY = os.environ.get('SEEDANCE_API_KEY')

BASE_URL   = 'https://ark.ap-southeast.bytepluses.com/api/v3'  # BytePlus international route (USD billing)
# Verify the current model id in the ModelArk console (see markdown note above) — override via .env if needed.
MODEL_ID   = os.environ.get('SEEDANCE_MODEL_ID', 'dreamina-seedance-2-0-260128')

RESOLUTION   = '720p'   # '480p' | '720p' | '1080p' | '2k'
DURATION_SEC = 5        # 4-15s per docs
RATIO        = '16:9'   # '16:9' | '9:16' | '4:3' | '3:4' | '21:9' | '1:1'
GENERATE_AUDIO = False

VIDEOS_PER_PROMPT = 1
POLL_INTERVAL_SEC = 10
MAX_POLL_ATTEMPTS = 60   # 60 * 10s = 10 min ceiling per job
MODEL_LABEL = 'Seedance'

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROMPTS_XLS = os.path.join(REPO_ROOT, 'prompts', 'all_prompts.xlsx')
OUTPUT_DIR  = os.path.join(REPO_ROOT, 'dataset', 'media', 'videos', MODEL_LABEL)
INDEX_CSV   = os.path.join(REPO_ROOT, 'dataset', 'labels', f'generated_{MODEL_LABEL}.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INDEX_CSV), exist_ok=True)

CREDENTIALS_OK = bool(SEEDANCE_API_KEY)
print(f'Model     : {MODEL_ID}')
print(f'Output    : {OUTPUT_DIR}')
print(f'Credentials present: {CREDENTIALS_OK}')
if not CREDENTIALS_OK:
    print('  -> Set SEEDANCE_API_KEY in .env before running the generation loop.')

In [ ]:
# ── Load prompts ─────────────────────────────────────────────────────────────
import pandas as pd

df_all = pd.read_excel(PROMPTS_XLS, sheet_name='All Prompts (100)')
df_all.columns = df_all.columns.str.strip()

# Video-compatible prompts = Modality in {'Video', 'Both'}
df_vid = df_all[df_all['Modality'].isin(['Video', 'Both'])].copy().reset_index(drop=True)

print(f'Total prompts    : {len(df_all)}')
print(f'Video-compatible : {len(df_vid)}')
print(df_vid.groupby(['Type', 'User Profile']).size().to_string())

In [ ]:
# ── Generation helpers (submit → poll → download) ───────────────────────────
import requests, time, hashlib, datetime

def auth_headers():
    return {'Authorization': f'Bearer {SEEDANCE_API_KEY}', 'Content-Type': 'application/json'}


def make_filename(prompt_id, prompt_type, index=0):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')
    h  = hashlib.sha256(f'{prompt_id}_{index}_{ts}'.encode()).hexdigest()[:10]
    t  = 'amb' if prompt_type == 'Ambiguous' else 'exp'
    return f'P{prompt_id:03d}_{t}_{h}.mp4'


def submit_task(prompt_text):
    """POST /contents/generations/tasks -> task id."""
    body = {
        'model': MODEL_ID,
        'content': [{'type': 'text', 'text': prompt_text}],
        'resolution': RESOLUTION,
        'duration': DURATION_SEC,
        'ratio': RATIO,
        'generate_audio': GENERATE_AUDIO,
    }
    resp = requests.post(f'{BASE_URL}/contents/generations/tasks', headers=auth_headers(), json=body, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data['id']


def extract_video_url(data):
    """Video URL location isn't 100% pinned down in public docs at the time of writing — check the
    common shapes reported for the ModelArk video-generation response and fail loudly if none match,
    so the raw payload can be inspected instead of silently mis-parsing it."""
    content = data.get('content')
    if isinstance(content, dict) and content.get('video_url'):
        return content['video_url']
    if isinstance(content, list) and content and isinstance(content[0], dict) and content[0].get('video_url'):
        return content[0]['video_url']
    if data.get('video_url'):
        return data['video_url']
    return None


def poll_task(task_id):
    """GET /contents/generations/tasks/{id} until succeeded/failed. Returns (status, video_url_or_msg)."""
    for attempt in range(MAX_POLL_ATTEMPTS):
        resp = requests.get(f'{BASE_URL}/contents/generations/tasks/{task_id}', headers=auth_headers(), timeout=30)
        resp.raise_for_status()
        data = resp.json()
        status = data.get('status')
        if status == 'succeeded':
            url = extract_video_url(data)
            if url:
                return 'ok', url
            print(f'  Warning: task {task_id} succeeded but no video_url found in response: {data}')
            return 'error', 'succeeded but no video_url in response'
        if status in ('failed', 'expired', 'cancelled'):
            return 'error', data.get('error', status)
        time.sleep(POLL_INTERVAL_SEC)
    return 'timeout', None


def download_video(url, dest_path):
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

print('Generation helpers ready.')

In [ ]:
# ── Resume: load already-generated index ─────────────────────────────────────
if os.path.exists(INDEX_CSV):
    df_done  = pd.read_csv(INDEX_CSV)
    done_ids = set(zip(df_done['prompt_id'], df_done['vid_index']))
    print(f'Resuming: {len(df_done)} videos already generated.')
else:
    df_done  = pd.DataFrame()
    done_ids = set()
    print('Starting fresh.')

In [ ]:
# ── Main generation loop ─────────────────────────────────────────────────────
from tqdm.notebook import tqdm

records = []

if not CREDENTIALS_OK:
    print('SEEDANCE_API_KEY not set in .env — skipping API calls.')
    print('Fill in .env (see .env.example) and re-run this cell to actually generate videos.')
else:
    for _, row in tqdm(df_vid.iterrows(), total=len(df_vid), desc=MODEL_LABEL):
        pid         = int(row['ID'])
        ptype       = row['Type']
        prompt_text = str(row['Prompt Text'])

        for vid_idx in range(VIDEOS_PER_PROMPT):
            if (pid, vid_idx) in done_ids:
                continue

            status = 'ok'
            fname  = ''
            try:
                task_id = submit_task(prompt_text)
                status, result = poll_task(task_id)
                if status == 'ok' and result:
                    fname = make_filename(pid, ptype, vid_idx)
                    download_video(result, os.path.join(OUTPUT_DIR, fname))
                    done_ids.add((pid, vid_idx))
                elif status == 'ok' and not result:
                    status = 'error'
            except Exception as e:
                print(f'  Error P{pid:03d}: {str(e)[:150]}')
                status = 'error'

            records.append({
                'file_path'   : f'videos/{MODEL_LABEL}/{fname}' if fname else '',
                'prompt_id'   : pid,
                'vid_index'   : vid_idx,
                'prompt_type' : ptype,
                'user_profile': row['User Profile'],
                'modality'    : row['Modality'],
                'category'    : row['Category'],
                'language'    : row['Language'],
                'status'      : status,
                'model'       : MODEL_LABEL,
                'prompt_text' : prompt_text,
            })

if records:
    df_new = pd.DataFrame(records)
    df_out = pd.concat([df_done, df_new], ignore_index=True) if not df_done.empty else df_new
    df_out.to_csv(INDEX_CSV, index=False)
    ok  = (df_out['status'] == 'ok').sum()
    err = (df_out['status'] == 'error').sum()
    print(f'\nDone. Generated: {ok} | Errors: {err}')
    print(f'Index: {INDEX_CSV}')
else:
    print('No records written this run.')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
if os.path.exists(INDEX_CSV):
    df_results = pd.read_csv(INDEX_CSV)
    print(f'=== {MODEL_LABEL} Generation Summary ===')
    print(df_results.groupby(['prompt_type', 'status']).size().unstack(fill_value=0).to_string())
    print(f"\nTotal videos saved: {(df_results['status']=='ok').sum()} / {len(df_results)}")
else:
    print('No index CSV yet — run the generation loop first.')